# Fase A — SFT LoRA del BARC-8B (de-risk de la infra)

Valida que podemos LoRA-fine-tunear el BARC-8B offline en Kaggle GPU con el formato de
training de BARC (chat `messages`, Llama-3.1). Objetivo: la loss baja + el adapter se guarda.
NO es entrenamiento real todavia (eso es la Fase C con data compositiva) — solo la infra.
Imagen BYOD (unsloth); bf16 (no hay bitsandbytes para 4-bit); LoRA en una L4.

In [ ]:
import os, glob, json, time
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
# localizar el modelo BARC (copia propia, montado via symlink)
mdir = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'config.json' in files and any(f.endswith(('.safetensors', '.bin')) for f in files):
        mdir = root; break
assert mdir, 'no se encontro el modelo BARC'
print('modelo en:', mdir)
# localizar el slice de data
djson = glob.glob('/kaggle/input/**/induction_heavy_slice.jsonl', recursive=True)
assert djson, 'no se encontro el slice de data'
dpath = djson[0]; print('data en:', dpath)

In [ ]:
from unsloth import FastLanguageModel
import torch
MAXLEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=mdir, full_finetuning=False, load_in_4bit=False,
    local_files_only=True, use_gradient_checkpointing='unsloth', max_seq_length=MAXLEN)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)
print('LoRA aplicado. chat_template presente:', tokenizer.chat_template is not None)

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files=dpath, split='train')
print('ejemplos:', len(ds))
def fmt(ex):
    return {'text': tokenizer.apply_chat_template(ex['messages'], tokenize=False)}
ds = ds.map(fmt, remove_columns=ds.column_names)
# filtrar por longitud para no reventar memoria
ds = ds.filter(lambda e: len(tokenizer(e['text'])['input_ids']) <= MAXLEN)
print('tras filtrar <=', MAXLEN, 'tok:', len(ds))
print('muestra[:300]:', ds[0]['text'][:300])

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
collator = DataCollatorForCompletionOnlyLM(
    response_template='<|start_header_id|>assistant<|end_header_id|>', tokenizer=tokenizer)
args = SFTConfig(
    output_dir='/kaggle/working/barc_sft_out', per_device_train_batch_size=1,
    gradient_accumulation_steps=8, warmup_steps=5, max_steps=40, learning_rate=1e-4,
    logging_steps=1, bf16=True, optim='adamw_8bit', lr_scheduler_type='cosine',
    seed=42, report_to='none', dataset_text_field='text', max_seq_length=MAXLEN,
    dataset_num_proc=2, packing=False)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
                     data_collator=collator, args=args)
t0 = time.time()
stats = trainer.train()
print('SFT OK en', round(time.time()-t0), 's | train_loss:', stats.training_loss)

In [ ]:
# guardar el adapter LoRA + reportar la curva de loss (validacion del de-risk)
model.save_pretrained('/kaggle/working/barc_lora_adapter')
tokenizer.save_pretrained('/kaggle/working/barc_lora_adapter')
losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
res = {'n_ejemplos': len(ds), 'max_steps': 40, 'loss_inicial': losses[0] if losses else None,
       'loss_final': losses[-1] if losses else None, 'bajo': (losses[0] > losses[-1]) if len(losses) > 1 else None}
json.dump(res, open('/kaggle/working/sft_result.json', 'w'), indent=1)
print('SFT DE-RISK:', json.dumps(res))
print('loss:', [round(x, 3) for x in losses])
print('adapter guardado en /kaggle/working/barc_lora_adapter')